Today you will learn:   

* What happens inside a retriever
* Difference between similarity_search() and Retriever
* What is k
* Similarity scores
* Why retrieval sometimes fails
* MMR (Max Marginal Relevance)
* Compare Normal Retrieval vs MMR
  
No Gemini import is needed today because today we're focusing only on retrieval, not answer generation.

User Question    
      ↓   
Convert Question → Embedding   
      ↓    
Search Vector Database    
      ↓   
Retrieve Best Chunks   
      ↓   
    Gemini  
      ↓   
Final Answer

In [1]:
import os

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

C:\Users\Dell\AppData\Local\Temp\ipykernel_8172\1814625698.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**Load all PDFs**

In [2]:
folder = "day16_pdfs"

documents = []

for file in os.listdir (folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(folder, file))
        documents.extend(loader.load())

print("Pages Loade:", len(documents))

Pages Loade: 121


**Check metadata**

In [3]:
print(documents[0].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'day16_pdfs\\attension-is-all-u-need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


**Split documents into chunks**

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 863


**Create embeddings**

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**Create FAISS vector  databse**

In [6]:
vector_db = FAISS.from_documents(
    chunks,
    embedding=embeddings
)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


**Save FAISS index**

In [7]:
vector_db.save_local("faiss_index")

print("FAISS Index Saved Successfully!")

FAISS Index Saved Successfully!


**Create normal Retriever**

In [8]:
retriever = vector_db.as_retriever(
    search_kwargs={"k":3}
)

print("Retriever Created Successfully!")

Retriever Created Successfully!


**Manual Similarity Search**

In [9]:
query = "What does BERT stand for?"

docs = vector_db.similarity_search(
    query,
    k=3
)

**Display Retrieved Chunks**

In [10]:
for i, doc in enumerate(docs):

    print("="*70)
    print(f"Document {i+1}")
    print("Source:", doc.metadata["source"])
    print()
    print(doc.page_content)

Document 1
Source: day16_pdfs\bert.pdf

asTi∈ RH.
For a given token, its input representation is
constructed by summing the corresponding token,
segment, and position embeddings. A visualiza-
tion of this construction can be seen in Figure 2.
3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we do not use traditional left-to-right or
right-to-left language models to pre-train BERT.
Instead, we pre-train BERT using two unsuper-
vised tasks, described in this section. This step
Document 2
Source: day16_pdfs\bert.pdf

ﬁne-tuning, since the [MASK] token does not ap-
pear during ﬁne-tuning. To mitigate this, we do
not always replace “masked” words with the ac-
tual [MASK] token. The training data generator
chooses 15% of the token positions at random for
prediction. If the i-th token is chosen, we replace
thei-th token with (1) the [MASK] token 80% of
the time (2) a random token 10% of the time (3)
the unchanged i-th token 10% of the time. Then,
Ti will be used t

**Trying differnt questions**

What is self-attention?

Who developed Gemini?

What is Transformer?

What is Machine Learning?

In [11]:
query = "What is self-attention?"

docs = vector_db.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(docs):

    print("="*70)
    print(f"Document {i+1}")
    print("Source:", doc.metadata["source"])
    print()
    print(doc.page_content)

Document 1
Source: day16_pdfs\attension-is-all-u-need.pdf

described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence. Self-attention has been
used successfully in a variety of tasks including reading comprehension, abstractive summarization,
textual entailment and learning task-independent sentence representations [4, 27, 28, 22].
Document 2
Source: day16_pdfs\attension-is-all-u-need.pdf

typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].
• The encoder contains self-attention layers. In a self-attention layer all of the keys, values
and queries come from the same place, in this case, the output of the previous layer in the
encoder. Each position in the encoder can attend to all positions in the previous layer of the
encoder.
• Similarly, self-attention layers in the decoder allow each position in t

In [12]:
query = "Who developed Gemini?"

docs = vector_db.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(docs):

    print("="*70)
    print(f"Document {i+1}")
    print("Source:", doc.metadata["source"])
    print()
    print(doc.page_content)

Document 1
Source: day16_pdfs\gemini.pdf

Gemini: A Family of Highly Capable Multimodal Models
Beroshi, Joel Moss, Jon Small, Jonathan Fildes, Kathy Meier-Hellstern, Lisa Patel, Oli Gaymond,
Rebecca Bland, Reena Jana, Tessa Lueth, and Tom Lue.
Our work is made possible by the dedication and efforts of numerous teams at Google. We would
like to acknowledge the support from Abhi Mohan, Adekunle Bello, Aishwarya Nagarajan, Alaa
Saade, Alejandro Lince, Alexander Chen, Alexander Kolbasov, Alexander Schiffhauer, Ameya Shringi,
Document 2
Source: day16_pdfs\gemini.pdf

Gemini: A Family of Highly Capable Multimodal Models
Terzi, Vladimir Mikulik, Igor Babuschkin, Aidan Clark, Diego de Las Casas, Aurelia Guy, Chris Jones,
James Bradbury, Matthew Johnson, Blake A. Hechtman, Laura Weidinger, Iason Gabriel, William S.
Isaac, Edward Lockhart, Simon Osindero, Laura Rimell, Chris Dyer, Oriol Vinyals, Kareem Ayoub,
Jeff Stanway, Lorrayne Bennett, Demis Hassabis, Koray Kavukcuoglu, and Geoffrey Irving.

In [13]:
query = "What is Transformer?"

docs = vector_db.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(docs):

    print("="*70)
    print(f"Document {i+1}")
    print("Source:", doc.metadata["source"])
    print()
    print(doc.page_content)

Document 1
Source: day16_pdfs\attension-is-all-u-need.pdf

Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
Document 2
Source: day16_pdfs\attension-is-all-u-need.pdf

The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-s

**Understanding K**

In [14]:
# k = 1
docs = vector_db.similarity_search(query, k=1)

for doc in docs:
    print(doc.metadata["source"])
    print(doc.page_content)

day16_pdfs\attension-is-all-u-need.pdf
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-


In [15]:
# k = 2
docs = vector_db.similarity_search(query, k= 2)

for doc in docs:
    print(doc.metadata["source"])
    print(doc.page_content)

day16_pdfs\attension-is-all-u-need.pdf
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
day16_pdfs\attension-is-all-u-need.pdf
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].


In [16]:
# K = 4
docs = vector_db.similarity_search(query, k=4)

for doc in docs:
    print(doc.metadata["source"])
    print(doc.page_content)

day16_pdfs\attension-is-all-u-need.pdf
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
day16_pdfs\attension-is-all-u-need.pdf
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].
day16

In [17]:
# k = 5
docs = vector_db.similarity_search(query, k=5)

for doc in docs:
    print(doc.metadata["source"])
    print(doc.page_content)

day16_pdfs\attension-is-all-u-need.pdf
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
day16_pdfs\attension-is-all-u-need.pdf
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].
day16

**Similarity Search with Scores**

In [18]:
results = vector_db.similarity_search_with_score(
    query,
    k=5
)

**Display Similarity SCore**

In [19]:
for doc, score in results:

    print("="*70)
    print("Score:", score)
    print("Source:", doc.metadata["source"])
    print()
    print(doc.page_content[:300])

Score: 0.96918344
Source: day16_pdfs\attension-is-all-u-need.pdf

Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder
Score: 1.0739497
Source: day16_pdfs\attension-is-all-u-need.pdf

The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in 
Score: 1.1078097
Source: day16_pdfs\bert.pdf

ture and the ﬁnal downstream architecture.
Model Architecture BERT’s model architec-
ture is a multi-layer bidirectional Transformer en-
coder based on the original implementation de-
scribed in Vaswani et al. (2017) and 

**Compare Different Questions**  
What is Gemini?

Who developed Gemini?

What is Transformer?

What is Machine Learning?

In [20]:
query = "What is Gemini?"

results = vector_db.similarity_search_with_score(
    query,
    k=5
)

for doc, score in results:

    print("="*70)
    print("Score:", score)
    print("Source:", doc.metadata["source"])
    print()
    print(doc.page_content[:300])

Score: 0.5432193
Source: day16_pdfs\gemini.pdf

Gemini: A Family of Highly Capable Multimodal Models
Beroshi, Joel Moss, Jon Small, Jonathan Fildes, Kathy Meier-Hellstern, Lisa Patel, Oli Gaymond,
Rebecca Bland, Reena Jana, Tessa Lueth, and Tom Lue.
Our work is made possible by the dedication and efforts of numerous teams at Google. We would
like
Score: 0.69254863
Source: day16_pdfs\gemini.pdf

Gemini: A Family of Highly Capable Multimodal Models
Terzi, Vladimir Mikulik, Igor Babuschkin, Aidan Clark, Diego de Las Casas, Aurelia Guy, Chris Jones,
James Bradbury, Matthew Johnson, Blake A. Hechtman, Laura Weidinger, Iason Gabriel, William S.
Isaac, Edward Lockhart, Simon Osindero, Laura Rimel
Score: 0.7800618
Source: day16_pdfs\gemini.pdf

Gemini: A Family of Highly Capable Multimodal Models
Daniel Salz, Mario Lucic, Michael Tschannen, Arsha Nagrani, Hexiang Hu, Mandar Joshi, Bo Pang,
Ceslee Montgomery, Paulina Pietrzyk, Marvin Ritter, AJ Piergiovanni, Matthias Minderer, Filip
Pavetic, Au

In [21]:
query = "Who developed Gemini?"

results = vector_db.similarity_search_with_score(
    query,
    k=5
)

for doc, score in results:

    print("="*70)
    print("Score:", score)
    print("Source:", doc.metadata["source"])
    print()
    print(doc.page_content[:300])

Score: 0.63992786
Source: day16_pdfs\gemini.pdf

Gemini: A Family of Highly Capable Multimodal Models
Beroshi, Joel Moss, Jon Small, Jonathan Fildes, Kathy Meier-Hellstern, Lisa Patel, Oli Gaymond,
Rebecca Bland, Reena Jana, Tessa Lueth, and Tom Lue.
Our work is made possible by the dedication and efforts of numerous teams at Google. We would
like
Score: 0.7118453
Source: day16_pdfs\gemini.pdf

Gemini: A Family of Highly Capable Multimodal Models
Terzi, Vladimir Mikulik, Igor Babuschkin, Aidan Clark, Diego de Las Casas, Aurelia Guy, Chris Jones,
James Bradbury, Matthew Johnson, Blake A. Hechtman, Laura Weidinger, Iason Gabriel, William S.
Isaac, Edward Lockhart, Simon Osindero, Laura Rimel
Score: 0.78886235
Source: day16_pdfs\gemini.pdf

Gemini: A Family of Highly Capable Multimodal Models
Daniel Salz, Mario Lucic, Michael Tschannen, Arsha Nagrani, Hexiang Hu, Mandar Joshi, Bo Pang,
Ceslee Montgomery, Paulina Pietrzyk, Marvin Ritter, AJ Piergiovanni, Matthias Minderer, Filip
Pavetic, A

In [22]:
query = "what is machine learning?"

results = vector_db.similarity_search_with_score(
    query,
    k=5
)

for doc, score in results:

    print("="*70)
    print("Score:", score)
    print("Source:", doc.metadata["source"])
    print()
    print(doc.page_content[:300])

Score: 1.0248727
Source: day16_pdfs\gemini.pdf

at Google and beyond. We build on many innovations in machine learning, data, infrastructure,
and responsible development – areas that we have been pursuing at Google for over a decade. The
models we present in this report provide a strong foundation towards our broader future goal to
develop a larg
Score: 1.1267996
Source: day16_pdfs\gemini.pdf

the need for more challenging and robust evaluations to measure their true understanding as the
current state-of-the-art LLMs saturate many benchmarks.
The Gemini family is a further step towards our mission to solve intelligence, advance science
and benefit humanity, and we are enthusiastic to see 
Score: 1.1816425
Source: day16_pdfs\gemini.pdf

training of large models.
ML Pathways is infrastructure software to support Google’s
efforts to build artificially intelligent systems capable of
generalizing across multiple tasks. This is specially suitable for
foundation models, including large langua

**Create normal retriever**

In [23]:
normal_retriever = vector_db.as_retriever(
    search_kwargs={"k":3}
)

**Test normal retriever**

In [24]:
query = "Who developed Gemini?"

normal_docs = normal_retriever.invoke(query)

**Display normal retriever output**

In [25]:
print("NORMAL RETRIEVER\n")

for i, doc in enumerate(normal_docs):

    print("="*70)
    print(f"Document {i+1}")
    print("Source:", doc.metadata["source"])
    print()
    print(doc.page_content)

NORMAL RETRIEVER

Document 1
Source: day16_pdfs\gemini.pdf

Gemini: A Family of Highly Capable Multimodal Models
Beroshi, Joel Moss, Jon Small, Jonathan Fildes, Kathy Meier-Hellstern, Lisa Patel, Oli Gaymond,
Rebecca Bland, Reena Jana, Tessa Lueth, and Tom Lue.
Our work is made possible by the dedication and efforts of numerous teams at Google. We would
like to acknowledge the support from Abhi Mohan, Adekunle Bello, Aishwarya Nagarajan, Alaa
Saade, Alejandro Lince, Alexander Chen, Alexander Kolbasov, Alexander Schiffhauer, Ameya Shringi,
Document 2
Source: day16_pdfs\gemini.pdf

Gemini: A Family of Highly Capable Multimodal Models
Terzi, Vladimir Mikulik, Igor Babuschkin, Aidan Clark, Diego de Las Casas, Aurelia Guy, Chris Jones,
James Bradbury, Matthew Johnson, Blake A. Hechtman, Laura Weidinger, Iason Gabriel, William S.
Isaac, Edward Lockhart, Simon Osindero, Laura Rimell, Chris Dyer, Oriol Vinyals, Kareem Ayoub,
Jeff Stanway, Lorrayne Bennett, Demis Hassabis, Koray Kavukcuoglu, an

**Create MMR retriever**

In [26]:
mmr_retriever = vector_db.as_retriever(

    search_type="mmr",

    search_kwargs={
        "k":3,
        "fetch_k":10
    }
)

**Test MMR Retriever**

In [27]:
mmr_docs = mmr_retriever.invoke(query)

**Display MMR retriever output**

In [28]:
print("MMR RETRIEVER\n")

for i, doc in enumerate(mmr_docs):

    print("="*70)
    print(f"Document {i+1}")
    print("Source:", doc.metadata["source"])
    print()
    print(doc.page_content)

MMR RETRIEVER

Document 1
Source: day16_pdfs\gemini.pdf

Gemini: A Family of Highly Capable Multimodal Models
Beroshi, Joel Moss, Jon Small, Jonathan Fildes, Kathy Meier-Hellstern, Lisa Patel, Oli Gaymond,
Rebecca Bland, Reena Jana, Tessa Lueth, and Tom Lue.
Our work is made possible by the dedication and efforts of numerous teams at Google. We would
like to acknowledge the support from Abhi Mohan, Adekunle Bello, Aishwarya Nagarajan, Alaa
Saade, Alejandro Lince, Alexander Chen, Alexander Kolbasov, Alexander Schiffhauer, Ameya Shringi,
Document 2
Source: day16_pdfs\gemini.pdf

each role does not indicate ordering of the contributions.
Gemini is a cross-Google effort, with members from Google DeepMind (GDM), Google Research
(GR), Bard/Assistant, Knowledge and Information (K&I), Core ML, Cloud, Labs, and more.
We thank Aakanksha Chowdhery, Dustin Tran, Heng-Tze Cheng, Jack W. Rae, Kate Olszewska,
Mariko Iinuma, Peter Humphreys, Shashi Narayan, and Steven Zheng for leading the preparation